# SAM Audio (Meta) — segmentacja / separacja dźwięku w Google Colab

SAM Audio to model Meta AI do izolowania konkretnego dźwięku z nagrania na podstawie opisu tekstowego, wskazówki wizualnej (wideo + maska) lub przedziału czasowego.

- Repozytorium: https://github.com/facebookresearch/sam-audio
- Blog: https://ai.meta.com/blog/sam-audio/
- Paper: https://arxiv.org/abs/2512.18099

**WAŻNE — dostęp do wag modelu:**
Checkpointy są "gated" na Hugging Face. Zanim uruchomisz ten notebook:
1. Wejdź na https://huggingface.co/facebook/sam-audio-large i kliknij *Request access* (akceptacja zwykle jest szybka/automatyczna).
2. Wygeneruj token dostępu (z uprawnieniem *Read*) na https://huggingface.co/settings/tokens
3. Wklej ten token w komórce logowania poniżej.

## Zanim zaczniesz
Ustaw akcelerator GPU: `Edit -> Notebook settings -> Hardware accelerator -> GPU (T4)`.

In [4]:
!nvidia-smi

Tue Sep  8 17:26:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Instalacja SAM Audio i zależności

In [5]:
!pip install -q 'git+https://github.com/facebookresearch/sam-audio.git'
!pip install -q huggingface_hub torchaudio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━

## 2. Logowanie do Hugging Face i wczytanie modelu

In [6]:
import os
import torch
import torchaudio
import numpy as np
from google.colab import userdata
from huggingface_hub import login, HfApi

print("NumPy Version:", np.__version__)

# 1. Retrieve Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# 2. Authenticate
login(token=hf_token)

# 3. Verify access
try:
    user_info = HfApi().whoami()
    print(f"\n✅ Success! Logged in as: {user_info['name']}")
    print(f"🔑 Token permission level: {user_info.get('auth', {}).get('accessToken', {}).get('role', 'unknown')}")
except Exception as e:
    print(f"\n❌ Authentication Error: {e}")
    print("Ensure 'HF_TOKEN' is added to Colab Secrets with 'Notebook access' enabled.")

# 4. Load SAM-Audio Model and Processor
from sam_audio import SAMAudio, SAMAudioProcessor

MODEL_ID = "facebook/sam-audio-large"

print(f"\nLoading {MODEL_ID}...")
processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
model = SAMAudio.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.eval().to(device)

print(f"✅ SAM-Audio successfully initialized on: {device.upper()}")

NumPy Version: 2.1.3


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.



✅ Success! Logged in as: Glitch55
🔑 Token permission level: fineGrained


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject